In [145]:
!pip install ucimlrepo
!pip install matplotlib
!pip install seaborn
!pip install plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [146]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [147]:
def write_variable_descriptions(variables: pd.DataFrame, output_path: str) -> None:
    lines = []

    if "name" in variables.columns and "description" in variables.columns:
        for _, row in variables.iterrows():
            name = row["name"]
            description = row["description"]

            if pd.isna(description):
                description = "No description available."

            lines.append(f"{name}:")
            lines.append(str(description).strip())
            lines.append("")
    else:
        # Fallback: store the full table if the expected columns are unavailable.
        lines.append(variables.to_string(index=False))

    Path(output_path).write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")


# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# variable information 
print(bank_marketing.variables) 

# variable information
write_variable_descriptions(bank_marketing.variables, "variable_information.txt")
print("Saved variable descriptions to variable_information.txt")

# store data in csv file, put X,y together
df = pd.concat([X, y], axis=1)
df.to_csv('bank_marketing.csv', index=False)

           name     role         type      demographic  \
0           age  Feature      Integer              Age   
1           job  Feature  Categorical       Occupation   
2       marital  Feature  Categorical   Marital Status   
3     education  Feature  Categorical  Education Level   
4       default  Feature       Binary              NaN   
5       balance  Feature      Integer              NaN   
6       housing  Feature       Binary              NaN   
7          loan  Feature       Binary              NaN   
8       contact  Feature  Categorical              NaN   
9   day_of_week  Feature         Date              NaN   
10        month  Feature         Date              NaN   
11     duration  Feature      Integer              NaN   
12     campaign  Feature      Integer              NaN   
13        pdays  Feature      Integer              NaN   
14     previous  Feature      Integer              NaN   
15     poutcome  Feature  Categorical              NaN   
16            

In [148]:
# Data Preprocessing Step 1: handle missing and inconsistent values

# Expected Values for each categorical column
expected_job = {
    'admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management',
    'retired', 'self-employed', 'services', 'student', 'technician',
    'unemployed', 'unknown'
}

expected_martial = {
    'single', 'married', 'divorced', 'unknown'
}

expected_education = {
    'primary', 'secondary', 'tertiary', 'unknown'
}

expected_contact = {
    'cellular', 'telephone', 'unknown'
}

expected_poutcome = {
    'unknown', 'other', 'failure', 'success'
}

binary_columns = {
    'yes', 'no'
}

# 1. Age: check if there are any missing values or inconsistent(non integer or negative)
# check if there are any missing values
missing_age = df['age'].isnull().sum()
print(f"Number of missing age values: {missing_age}")

# check if there are any inconsistent values (negative or non integer)
inconsistent_age = df[~df['age'].isin(range(0, 100))]
print(f"Number of inconsistent age values: {len(inconsistent_age)}")

# 2. Job: check if there are any missing values    
# or inconsistent(null values or unexpected job)
missing_job = df['job'].isnull().sum()
print(f"Number of missing job values: {missing_job}")

# check how many non-null values are expected
expected_job = df['job'].isin(expected_job).sum()
print(f"Number of expected job values: {expected_job}")

# replace the null job column with 'unknown' tag
df['job'] = df['job'].fillna('unknown')

# 3. Martial: check if there are any missing values or 
# inconsistent(non string or unexpected martial)

# check how many martial column is missing
missing_martial = df['marital'].isnull().sum()
print(f"Number of missing martial values: {missing_martial}")

# check how many non-null values are expected
expected_martial = df['marital'].isin(expected_martial).sum()
print(f"Number of expected martial values: {expected_martial}")

# 4. Education: check if there are any missing values or 
# inconsistent(non string or unexpected education)

# check how many education column is missing
missing_education = df['education'].isnull().sum()
print(f"Number of missing education values: {missing_education}")

# check how many non-null values are expected
expected_education = df['education'].isin(expected_education).sum()
print(f"Number of expected education values: {expected_education}")

# replace the missing education column with 'unknown' tag
df['education'] = df['education'].fillna('unknown')

missing_education = df['education'].isnull().sum()
print(f"Number of missing education values: {missing_education}")

# 5. Default: check if there are any missing values or 
# inconsistent(non string or unexpected default)

# check how many default column is missing
missing_default = df['default'].isnull().sum()
print(f"Number of missing default values: {missing_default}")

# check how many non-null values are expected
expected_default = df['default'].isin(binary_columns).sum()
print(f"Number of expected default values: {expected_default}")

# 6. Balance: check if there are any missing values or 
# inconsistent(non integer or negative)

# check if there are any missing values
missing_balance = df['balance'].isnull().sum()
print(f"Number of missing balance values: {missing_balance}")

# check if there are any non-integer, it can be negative
non_integer_balance = df[~df['balance'].isin(range(-10000000, 10000000))]
print(f"Number of non-integer balance values: {len(non_integer_balance)}")

if len(non_integer_balance) > 0:
    print(f"There are {len(non_integer_balance)} non-integer balance values")
else:
    print("There are no non-integer balance values")

# 7. Housing: check if there are any missing values or 
# inconsistent(non string or unexpected housing)

# check how many housing column is missing
missing_housing = df['housing'].isnull().sum()
print(f"Number of missing housing values: {missing_housing}")

# check how many non-null values are expected
expected_housing = df['housing'].isin(binary_columns).sum()
print(f"Number of expected housing values: {expected_housing}")

# 8. Loan: check if there are any missing values or 
# inconsistent(non string or unexpected loan)

# check how many loan column is missing
missing_loan = df['loan'].isnull().sum()
print(f"Number of missing loan values: {missing_loan}")

# check how many non-null values are expected
expected_loan = df['loan'].isin(binary_columns).sum()
print(f"Number of expected loan values: {expected_loan}")

# 9. Contact: check if there are any missing values or 
# inconsistent(non string or unexpected contact)
# check how many contact column is missing (NaN or empty strings)
missing_contact = (df['contact'].isnull() | (df['contact'] == '')).sum()
print(f"Number of missing contact values: {missing_contact}")

# replace the missing contact column with 'unknown' tag
df['contact'] = df['contact'].replace('', 'unknown').fillna('unknown')

# verify the final count of 'expected' values including 'unknown'
expected_contact_count = df['contact'].isin(expected_contact).sum()
print(f"Number of expected contact values: {expected_contact_count}")

# verify no more missing values
missing_contact_after = (df['contact'].isnull() | (df['contact'] == '')).sum()
print(f"Number of missing contact values after filling: {missing_contact_after}")

# 10. Day_of_week & Month: check if there are any missing values or 
# inconsistent(non string or unexpected day_of_week or month)
# check missing values first
missing_day_of_week = df['day_of_week'].isnull().sum()
print(f"Number of missing day_of_week values: {missing_day_of_week}")

missing_month = df['month'].isnull().sum()
print(f"Number of missing month values: {missing_month}")

# then check if the date is valid since each month has a different number of days
# we can use the datetime module to check if the date is valid

expected_month = {
    'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'
}

expected_day_of_month = {
    'jan': 31, 'feb': 28, 'mar': 31, 'apr': 30, 'may': 31, 'jun': 30, 
    'jul': 31, 'aug': 31, 'sep': 30, 'oct': 31, 'nov': 30, 'dec': 31
}

# First check if the month is in expected_month
missing_month = df['month'].isin(expected_month).sum()
print(f"Number of missing month values: {missing_month}")

# Given the month of each row, check if the day of month is valid
# for each item, retrive month and set range (1, expected_day_of_month[month] + 1)
# check if the day of month is in the range
invalid_day_of_month = 0
for index, row in df.iterrows():
    month = row['month']
    day_of_month = row['day_of_week']
    if month in expected_day_of_month:
        if day_of_month > expected_day_of_month[month]:
            print(f"Invalid day of month for {month}: {day_of_month}")
            invalid_day_of_month += 1

print(f"Number of invalid day of month: {invalid_day_of_month}")

# 11. Duration: check if there are any missing values or 
# inconsistent(non integer or negative)

# check if there are any missing values
missing_duration = df['duration'].isnull().sum()
print(f"Number of missing duration values: {missing_duration}")

# check if there are any non-integer, it can be negative
non_integer_duration = df[~df['duration'].isin(range(0,1000000))]
print(f"Number of non-integer duration values: {len(non_integer_duration)}")

if len(non_integer_duration) > 0:
    print(f"There are {len(non_integer_duration)} non-integer or negative duration values")
    print(non_integer_duration)
else:
    print("There are no non-integer duration values")

# 12. Campaign: check if there are any missing values or
# inconsistent(non integer or negative)
# campaign: check if there are any missing values or 
# inconsistent(non integer or negative)

# check if there are any missing values
missing_campaign = df['campaign'].isnull().sum()
print(f"Number of missing campaign values: {missing_campaign}")

# check if there are any non-integer, it can be negative
non_integer_campaign = df[~df['campaign'].isin(range(0,1000000))]
print(f"Number of non-integer campaign values: {len(non_integer_campaign)}")

if len(non_integer_campaign) > 0:
    print(f"There are {len(non_integer_campaign)} non-integer campaign values")
    print(non_integer_campaign)
else:
    print("There are no non-integer campaign values")

# 13. Pdays: check if there are any missing values or 
# inconsistent(non integer or negative)
# check if there are any missing values
missing_pdays = df['pdays'].isnull().sum()
print(f"Number of missing pdays values: {missing_pdays}")

# check if there are any non-integer, it can be negative
non_integer_pdays = df[~df['pdays'].isin(range(-1, 1000000))]
print(f"Number of non-integer pdays values: {len(non_integer_pdays)}")

if len(non_integer_pdays) > 0:
    print(f"There are {len(non_integer_pdays)} non-integer or negative pdays values")
    # print(non_integer_pdays)
else:
    print("There are no non-integer pdays values")

# 14. Previous: check if there are any missing values or 
# inconsistent(non integer or negative)
# check if there are any missing values
missing_previous = df['previous'].isnull().sum()
print(f"Number of missing previous values: {missing_previous}")

# check if there are any non-string values
non_string_previous = df[~df['previous'].isin(range(0,1000000))]

if len(non_string_previous) > 0:
    print(f"There are {len(non_string_previous)} non-string previous values")
    print(non_string_previous)
else:
    print("There are no non-string previous values")

# 15. Poutcome: check if there are any missing values or 
# inconsistent(non string or unexpected poutcome)
# check if there are any missing values
missing_poutcome = df['poutcome'].isnull().sum()
print(f"Number of missing poutcome values: {missing_poutcome}")

# check if there are any non-string values
non_string_poutcome = df[~df['poutcome'].isin(expected_poutcome)]

if len(non_string_poutcome) > 0:
    print(f"There are {len(non_string_poutcome)} non-string poutcome values")
    # print(non_string_poutcome)
else:
    print("There are no non-string poutcome values")

# replace the missing poutcome values with 'unknown' tag
df['poutcome'] = df['poutcome'].fillna('unknown')

# 16. Target: check if there are any missing values or 
# inconsistent(non string or unexpected target)
# check if there are any missing values
missing_target = df['y'].isnull().sum()
print(f"Number of missing target values: {missing_target}")

# check if there are any non-string values
non_string_target = df[~df['y'].isin(binary_columns)]

if len(non_string_target) > 0:
    print(f"There are {len(non_string_target)} unexpected target values")
else:
    print("There are no unexpected target values")

Number of missing age values: 0
Number of inconsistent age values: 0
Number of missing job values: 288
Number of expected job values: 44923
Number of missing martial values: 0
Number of expected martial values: 45211
Number of missing education values: 1857
Number of expected education values: 43354
Number of missing education values: 0
Number of missing default values: 0
Number of expected default values: 45211
Number of missing balance values: 0
Number of non-integer balance values: 0
There are no non-integer balance values
Number of missing housing values: 0
Number of expected housing values: 45211
Number of missing loan values: 0
Number of expected loan values: 45211
Number of missing contact values: 13020
Number of expected contact values: 45211
Number of missing contact values after filling: 0
Number of missing day_of_week values: 0
Number of missing month values: 0
Number of missing month values: 45211
Number of invalid day of month: 0
Number of missing duration values: 0
Number

In [149]:
# Data Preprocessing Step 2: Encoding Categorical Variables

# 1. Binary Encoding (0/1) for: default, housing, loan, and the target y.
# Binary Encoding is used for features with only two categories to save space and maintain simplicity.
binary_mapping = {'yes': 1, 'no': 0}
df['default'] = df['default'].map(binary_mapping)
df['housing'] = df['housing'].map(binary_mapping)
df['loan'] = df['loan'].map(binary_mapping)
df['y'] = df['y'].map(binary_mapping)

# 2. One-Hot Encoding for nominal variables: job, marital, education, contact, month, and poutcome.
nominal_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df = pd.get_dummies(df, columns=nominal_cols, dtype=int)

# Verify the encoding
print("Final Columns:", df.columns.tolist())
df.head()

# store the encoded table in a new csv file
df.to_csv('encoded_bank_marketing.csv', index=False)


Final Columns: ['age', 'default', 'balance', 'housing', 'loan', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'y', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown', 'marital_divorced', 'marital_married', 'marital_single', 'education_primary', 'education_secondary', 'education_tertiary', 'education_unknown', 'contact_cellular', 'contact_telephone', 'contact_unknown', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_failure', 'poutcome_other', 'poutcome_success', 'poutcome_unknown']


In [ ]:
# Data Preprocessing Step 3: Optional New Features
# For now, we skip this step.

# 1. We need to combine day_of_week and month to create a new feature called day_of_year
# day_of_year is the number of days since the start of the year
# we can use the datetime module to do this
df['day_of_year'] = df['day_of_week'] + df['month']

# 2. We need to create a new feature called day_of_month


In [151]:
# Data Preprocessing Step 4: Data Splitting (70/15/15)
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (which will be split into val and test)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# Second split: 50% of temp_df for validation, 50% for test (0.5 * 0.3 = 0.15 each)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Save the split data to CSV files
train_df.to_csv('encoded_train_bank_marketing.csv', index=False)
val_df.to_csv('encoded_val_bank_marketing.csv', index=False)
test_df.to_csv('encoded_test_bank_marketing.csv', index=False)

# print size and shape of the split data
print("Training set size:", train_df.shape)
print("Validation set size:", val_df.shape)
print("Test set size:", test_df.shape)


Training set size: (31647, 49)
Validation set size: (6782, 49)
Test set size: (6782, 49)


In [ ]:
# Data Preprocessing Step 5: Feature Scaling



Scaled campaign mean (train): -0.0000
Scaled campaign std (train): 1.0000


,age,default,balance,housing,loan,day_of_week,duration,campaign,pdays,previous,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
10747,-0.464799,0,-0.456185,0,0,0.140619,-0.404011,0.987486,-0.410655,-0.240512,...,1,0,0,0,0,0,0,0,0,1
26054,1.416343,0,-0.390344,0,0,0.380915,0.210292,0.576326,-0.410655,-0.240512,...,0,0,0,1,0,0,0,0,0,1
9125,0.475772,0,-0.456185,1,0,-1.301157,-0.674459,-0.014546,-0.410655,-0.240512,...,1,0,0,0,0,0,0,0,0,1
41659,0.005486,0,0.694704,0,0,-1.781749,0.171657,-1.001992,0.781281,1.812098,...,0,0,0,0,1,0,0,0,1,0
4443,-0.276685,0,-0.456185,1,0,0.501063,-0.647414,-1.001992,-0.410655,-0.240512,...,0,0,1,0,0,0,0,0,0,1
